# Reproducing Li et al. (2015) from dandiset 000010

Paper. Li N, Chen T-W, Guo ZV, Gerfen CR, Svoboda K (2015). A motor cortex circuit for motor
planning and movement. *Nature* 519, 51-56.
[doi:10.1038/nature14178](https://doi.org/10.1038/nature14178). The claims file
(`claims/000010/10.1038-nature14178.yaml`) was written from the version of record (main text,
Methods and Extended Data legends) and checked by a second reader.

Dandiset. [DANDI:000010](https://dandiarchive.org/dandiset/000010/0.220126.1905), version
0.220126.1905. The paper cites no DANDI version (it predates the deposit and names the CRCNS
alm-1 data). This is the only published version, and so the most recent one; the draft lists the
same 158 assets and total size (its later modification was not examined further). The analysis
streams the published version.

What the dandiset holds. 99 extracellular sessions from 19 mice (left ALM, sorted units, trial
tables, behavioural events, a lick trace and, in 75 sessions, the laser power of the optogenetic
tagging light delivered during behavioural trials) and 59 two-photon imaging planes from 4 mice
(raw and neuropil fluorescence, ROI masks, PT/IT labels, trial tables). It holds no data from the
photoinhibition and photoactivation experiments and no histology.

Questions. Seven questions were asked of a reader (`questions/000010.yaml`); the claims they
touch are tested under every reading or, for the reaction time and the antidromic latencies,
blocked until the answer.

| Claim | Tested | Why not |
|---|---|---|
| behaviour-performance | yes | |
| fig1d-delay-inhibition-ipsi-bias | no | data missing: no photoinhibition mice or photostimulation trials |
| fig1d-sample-inhibition-smaller | no | data missing (as above) |
| units-classification | yes | |
| fig2-selective-fraction | yes | |
| fig2d-preparatory-perimovement | yes | |
| fig2c-selectivity-time-course | yes | |
| fig2c-population-dynamics | yes | (outcome depends on details the claims file leaves open) |
| ed2c-error-trials-predict-choice | yes | |
| fig2e-no-contra-bias-spikes | yes | |
| ed1-fast-spiking-properties | yes | |
| tagging-counts | yes, in part | the failed-collision classes carry no label |
| antidromic-latencies | no | methods underspecified: no rule for picking out the antidromic spike (question asked); a variant is shown |
| fig3e-pt-it-both-activity-types | yes | |
| fig3f-pt-contra-preferring | yes | (outcome depends on whether trials with tagging light are kept) |
| fig3f-pt-contra-selectivity | yes | |
| imaging-selectivity-counts | yes | |
| fig4f-spatially-intermingled | yes | |
| fig4h-pt-contra-imaging | yes | no plane depth, so no layer 5 selection |
| reaction-time | no | methods underspecified: no rule for reading a reaction time from the lick trace (question asked) |
| fig5b-pt-delay-activation-contra-bias | no | data missing: no photostimulation trials |
| fig5b-pt-sample-activation-contra-bias | no | data missing (as above) |
| ed6b-pt-dose-response | no | data missing (as above) |
| ed6d-it-activation-inconsistent | no | data missing (as above) |
| ed3e-high-power-contra-licking | no | data missing (as above) |
| ed7-pt-activation-persistent | no | data missing (as above) |
| ed3-anatomy-it-pt-separate | no | data missing: no histology |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import alm

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
assets = alm.list_assets()
meta = alm.version_metadata()
E = alm.load_all(assets, "ephys")
O = alm.load_all(assets, "ophys")
print(f"dandiset {alm.DANDISET} version {alm.VERSION}: {len(assets)} assets; loaded {len(E)} extracellular "
      f"sessions and {len(O)} imaging planes in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Archive summary:", meta["assetsSummary"]["numberOfSubjects"], "subjects,",
      meta["assetsSummary"]["numberOfFiles"], "files; approaches:",
      [a["name"] for a in meta["assetsSummary"]["approach"]])
print("Related resource:", [r["identifier"] for r in meta.get("relatedResource", [])])
print(assets.groupby("kind").agg(files=("path", "size"), subjects=("subject", "nunique"),
                                 GB=("size", lambda x: round(x.sum() / 1e9, 1))).to_string())
info = pd.DataFrame([dict(kind="ephys", subject=s["subject"], genotype=s["genotype"], region=",".join(s["region"]),
                          optogenetics=",".join(s["optogenetics"])) for s in E] +
                    [dict(kind="ophys", subject=s["subject"], genotype=s["genotype"], region=s["plane_location"],
                          optogenetics="") for s in O])
print("\nSessions per subject, with genotype, recording region and optogenetic sites named in the files:")
print(info.groupby(["kind", "subject", "genotype", "region", "optogenetics"]).size().rename("sessions").to_string())
print("\nData objects in the files (acquisition, stimulus, processing, analysis):")
print(pd.Series([g for s in E + O for g in s["groups"]]).value_counts().to_string())
print("\nTrial table columns:", sorted({tuple(s["trial_colnames"]) for s in E + O}))
print("Unit table columns:", sorted({tuple(s["unit_colnames"]) for s in E}))
print("ROI table columns:", sorted({tuple(s["roi_colnames"]) for s in O}))
print("Imaging plane descriptions:", sorted({s["plane_description"] for s in O}))

The files hold 99 extracellular sessions from 19 mice and 59 imaging planes from 4 mice, the
numbers of the paper (99 sessions and 19 mice; 59 fields of view and 4 mice). Every
extracellular session is in left ALM ("left M2"). The only stimulus records are the laser power and
AOM input traces of the tagging light in 75 sessions (next cells); the trial tables have no column
for photostimulation. The VGAT-ChR2-EYFP and PV-ires-Cre x ReaChR mice of Figure 1 are not among
the subjects, and the three Ai32 crosses (the photoactivation lines of Figure 5) have no stimulus
record, so no claim on photoinhibition or photoactivation can be tested. The imaging mice are given the
genotype "Rosa-lsl-tdtomato"; the Methods use C57Bl/6Crl mice. No imaging plane gives its depth.

In [ ]:
rows = []
for kind, S in (("ephys", E), ("ophys", O)):
    for i, s in enumerate(S):
        tr = s["trials"].assign(kind=kind, session=i, subject=s["subject"])
        rows.append(tr)
T = pd.concat(rows, ignore_index=True)
T["matched"] = T["sample"].notna()
print("Trials by kind, instruction, outcome and early-lick flag; `matched` counts trials with a sample event "
      f"{alm.MATCH_WIN[0]} to {alm.MATCH_WIN[1]} s after the trial start:")
print(T.groupby(["kind", "instruction", "outcome", "early"]).matched.agg(trials="size", matched="sum").to_string())
off = (T["sample"] - T.start)[T.matched]
print("\nSample event minus trial start (s), by kind:")
print(off.groupby(T.kind[T.matched]).describe().round(4).to_string())
n_ev = {k: sum(s["n_events"]["sample"] for s in S) for k, S in (("ephys", E), ("ophys", O))}
print("\nSample events in the files:", n_ev, "; matched to a trial:", T.groupby("kind").matched.sum().to_dict())

# Performing trials without events: their spikes
unm = []
for i, s in enumerate(E):
    tr = s["trials"]
    m = tr["sample"].isna() & tr.instruction.isin(["left", "right"])
    if m.any():
        allsp = np.sort(np.concatenate(s["spikes"]))
        n = alm.counts(allsp, tr.start[m].values, tr.stop[m].values)
        unm.append(pd.DataFrame(dict(session=i, subject=s["subject"], instruction=tr.instruction[m],
                                     outcome=tr.outcome[m], early=tr.early[m], spikes=n,
                                     dur=(tr.stop - tr.start)[m].round(2))))
unm = pd.concat(unm)
print(f"\nExtracellular trials with a lick instruction but no events: {len(unm)} in {unm.session.nunique()} sessions")
print(unm.groupby(["instruction", "outcome", "early"]).agg(trials=("spikes", "size"), with_spikes=("spikes", lambda x: int((x > 0).sum())),
      max_spikes=("spikes", "max"), lasting_5_5_s=("dur", lambda x: int((x == 5.5).sum()))).to_string())
print("Per session, fraction of lick-instruction trials without events (top 10):")
frac = unm.groupby("session").size() / pd.Series({i: s["trials"].instruction.isin(["left", "right"]).sum() for i, s in enumerate(E)})
print(frac.dropna().sort_values(ascending=False).head(10).round(3).to_string())

d = (T.go - T["sample"])
ne = T.early.isin(["no early", "False"]) & T.matched & T.instruction.isin(["left", "right"])
print("\nGo cue minus sample event (s) in trials without an early lick:")
print(d[ne].describe().round(4).to_string())
print("Trials without an early lick whose go cue is not 2.6 s after the sample event (dropped):",
      int((ne & ((d - 2.6).abs() >= 0.01)).sum()), "of", int(ne.sum()))
print("Go cue minus sample event (s) in early-lick trials:", d[T.early.eq("early") & T.matched].describe().round(2).to_dict())
print("Delay event minus sample event (s), median by kind:", (T.delay - T["sample"]).groupby(T.kind).median().round(3).to_dict())
print("\nEarly-lick flags in the imaging files:", T[T.kind == "ophys"].early.value_counts().to_dict())

Trials are matched to their behavioural events by the sample event, which falls 0.572 s
(extracellular) or 0.583 s (imaging) after the trial start in almost every trial. In the
extracellular files, the trials that carry no events are the "non-performing" trials and 1,540
trials with a lick instruction. Of these, 1,447 are marked "right", "hit" and "no early", last
exactly 5.5 s, and 1,345 of them hold no spike from any unit; in some sessions such trials are a
quarter of the trials. They look like placeholders for trials that were removed (photostimulation
trials, deployed on 25% of trials, would fit), filled with default values. The other 93 hold
spikes but no sample event in the matching window. All 1,540 are left out of every analysis
here, the behaviour included; counted as given, the placeholders would raise performance. In trials
without an early lick, the go cue follows the sample event by 2.6 s, the 1.3 s sample and 1.3 s
delay epochs of the Methods; 66 extracellular trials break this and are dropped. In early-lick
trials the go cue comes later, consistent with the delay being restarted after an early lick. The
imaging files flag no trial as an early lick, so early licks cannot be excluded there.

In [ ]:
rows = []
for i, s in enumerate(E):
    tr = s["trials"]
    for k, v in s["laser"].items():
        on = np.sort(v["onsets"])
        rows.append(dict(session=i, subject=s["subject"], genotype=s["genotype"], series=k, unit=v["unit"],
                         max=round(v["max"], 1), pulses=len(on), segments=v["n_segments"],
                         width_ms=np.round(np.median(v["width_ms"]), 2) if len(v["width_ms"]) else np.nan,
                         pair_gap_ms=round(1000 * np.median(np.diff(on)[np.diff(on) < 0.05]), 1) if len(on) > 1 else np.nan,
                         pair_period_s=round(np.median(np.diff(alm.pair_onsets(on))), 3) if len(on) > 2 else np.nan,
                         light_trials=int(tr.light.sum()), trials=len(tr), outside=s["pulses_outside_trials"]))
LZ = pd.DataFrame(rows)
print("Sessions with a laser power series:", LZ.session.nunique(), "of", len(E))
print(LZ.groupby(["genotype", "series", "unit", "max"]).agg(sessions=("session", "size"), pulses=("pulses", "sum"),
      light_trials=("light_trials", "sum"), trials=("trials", "sum"), width_ms=("width_ms", "median"),
      pair_gap_ms=("pair_gap_ms", "median"), pair_period_s=("pair_period_s", "median"), outside=("outside", "sum")).to_string())
print("Genotypes without any stimulus series:", sorted({s["genotype"] for s in E} - set(LZ.genotype)))
seg = sorted({int(x) for s in E for v in s["laser"].values() for x in v["segment_samples"]})
print("Samples per stored segment of the laser series:", seg, "; rate (Hz):", sorted({round(v["rate"]) for s in E for v in s["laser"].values()}))
TL = pd.concat([s["trials"].assign(session=i) for i, s in enumerate(E)], ignore_index=True)
print("\nTrials with light, by outcome (extracellular, trials with events):")
print(pd.crosstab([TL.outcome[TL["sample"].notna()]], TL.light[TL["sample"].notna()]).to_string())
pw = np.concatenate([v["power"] for s in E for v in s["laser"].values()])
print("Laser power at the pulses (file units):", np.unique(np.round(pw)).tolist())

In 75 sessions (the virus-injected Sim1, Rbp4 and Tlx mice; 84 series, as the Rbp4 sessions have
one for each site), `stimulus/presentation` holds a laser power series stored in 5 s segments at
10 kHz, one per trial with light (10 s where two such trials follow without a gap). The two Tlx x Ai32 mice, whose 8 IT units the files
label, have no light record. The pulses are
1 ms wide, in pairs 10 ms apart, every 0.5 s, at 82 (left pons) or 47.1 (bilateral ALM): the
antidromic tagging protocol of the Methods ("Pairs of laser pulses (1 ms duration, 47-82 mW peak
power, separated by 10 ms) were deployed every 500ms"). The series are labelled "watts", but the
values match the Methods' milliwatts. The light was given during behavioural trials, which the
trial tables do not flag; here a trial has light when a pulse falls in its window. The claims file
does not say whether these trials belong in the task analyses (question asked); the
electrophysiology claims are run with and without them. The behaviour claim asks for trials
without photostimulation, so it leaves them out.

In [ ]:
U = alm.unit_table(E)
print("Deposited units:", len(U), "; quality labels:", pd.Series(np.concatenate([s["quality"] for s in E])).value_counts().to_dict())
print("Sampling rate of the waveforms (Hz):", sorted(set(np.concatenate([s["unit_rate"] for s in E]))),
      "; waveform length (samples):", sorted({s["waveform"].shape[1] for s in E}))
print("\nUnits by file label, width class (trough to peak of the mean waveform) and duplicate flag:")
print(U.groupby(["label", "width_class", "is_duplicate"]).size().to_string())
print("\nEvery PT or IT unit has an identical spike train in a unit labelled Pyr in the same file:",
      int(U.is_duplicate.sum()), "of", int(U.label.isin(["PT", "IT"]).sum()))
print("Spike widths (ms) by label:")
print(U.groupby("label").width.describe().round(3).to_string())
print("Units whose waveform peak after the trough is the last sample:", int(U.peak_at_end.sum()),
      f"(the waveform ends {(28 - 15) / 19531.25 * 1000:.3f} ms after the trough at sample 15)")
print("Trough sample index:", sorted({int(np.argmin(w)) for s in E for w in s["waveform"]}))
spans = np.concatenate([np.diff(s["obs_intervals"], axis=1).ravel() for s in E])
print("\nObservation intervals per unit: lengths (s)", np.unique(spans.round(3)))
inside = []
for s in E:
    tr = s["trials"]
    allsp = np.sort(np.concatenate(s["spikes"]))
    k = np.searchsorted(tr.start.values, allsp, side="right") - 1
    inside.append(np.mean((k >= 0) & (allsp <= tr.stop.values[np.clip(k, 0, None)] + 1e-6)))
print("Fraction of spikes inside a trial window [start, stop], per session: min", round(min(inside), 4),
      "median", round(float(np.median(inside)), 4))

The deposited units are 1,245 labelled "Pyr", 124 "FS", 45 "PT" and 27 "IT". Every PT and IT unit
is a second copy of a unit labelled Pyr in the same file (identical spike trains), so the
recordings hold 1,369 distinct units; duplicates are counted once, and the PT/IT label is carried
to its Pyr twin. The spike widths computed from the waveforms put every Pyr unit above 0.45 ms
and every FS unit below 0.35 ms, so the labels agree with the width rule, and no unit has an
intermediate width: the 39 intermediate units of the Methods are not in the dandiset. The
waveforms are 29 samples with the trough at sample 15, so widths above 0.666 ms cannot be
measured; this does not affect the classification at 0.35 and 0.45 ms. Spikes are stored only
within the 5.5 s trial windows, from 0.57 s before the sample event to 2.33 s after the go cue in
trials without an early lick, which covers every epoch used here.

In [ ]:
# Clock check, extracellular: selectivity of each session's putative pyramidal units before the
# sample event (where no task information exists) and in the response epoch.
P = U[(U.width_class == "Pyr") & ~U.is_duplicate]
rows = []
for si in sorted(P.session.unique()):
    Ps = P[P.session == si]
    pre = alm.window_counts(E, Ps, [-0.5], width=0.5)
    resp = alm.window_counts(E, Ps, [2.6], width=1.3)
    p_pre = [alm.ttest_p(R[:, 0], L[:, 0]) for R, L in pre.values()]
    p_resp = [alm.ttest_p(R[:, 0], L[:, 0]) for R, L in resp.values()]
    rows.append(dict(session=si, subject=E[si]["subject"], units=len(Ps),
                     sel_pre=np.mean(np.array(p_pre) < 0.05), sel_resp=np.mean(np.array(p_resp) < 0.05)))
clock = pd.DataFrame(rows)
print("Fraction of units with lick right vs lick left t-test P < 0.05, correct trials, over all sessions:")
print(clock[["sel_pre", "sel_resp"]].describe().round(3).to_string())
print("Sessions where the response-epoch fraction does not exceed the pre-sample fraction:")
print(clock[clock.sel_resp <= clock.sel_pre].to_string(index=False))

In [ ]:
# Clock check, imaging: rank sum test between trial types before the sample event and in the
# response epoch, per plane, included ROIs, correct trials.
rows = []
for i, pl in enumerate(O):
    X, rel, right, dropped = alm.plane_trials(pl, "correct")
    inc = pl["included"]
    pre = X[:, (rel >= -0.5) & (rel < 0), :][:, :, inc].mean(1)
    resp = X[:, (rel >= 2.6) & (rel < 3.9), :][:, :, inc].mean(1)
    rows.append(dict(plane=i, subject=pl["subject"], rois=int(inc.sum()), trials=len(right), dropped=dropped,
                     sel_pre=np.mean(stats.ranksums(pre[right], pre[~right], axis=0).pvalue < 0.05),
                     sel_resp=np.mean(stats.ranksums(resp[right], resp[~right], axis=0).pvalue < 0.05)))
iclock = pd.DataFrame(rows)
print("Fraction of included ROIs with rank sum P < 0.05:")
print(iclock[["sel_pre", "sel_resp"]].describe().round(3).to_string())
print("Planes where the response-epoch fraction does not exceed the pre-sample fraction:")
print(iclock[iclock.sel_resp <= iclock.sel_pre].to_string(index=False))
print("Trials dropped for lack of 0.5 s of frames before the trial start:", int(iclock.dropped.sum()))

In [ ]:
print("Imaging rate (Hz):", sorted({round(pl["rate"], 3) for pl in O}), "; first frame time (s):",
      sorted({round(pl["t0"], 4) for pl in O}))
dur = [(pl["t0"] + len(pl["F"]) / pl["rate"], pl["trials"].stop.max()) for pl in O]
print("Last frame time minus last trial stop (s): min", round(min(a - b for a, b in dur), 2), "max", round(max(a - b for a, b in dur), 2))
roi = pd.DataFrame(dict(type=np.concatenate([pl["roi_type"] for pl in O]),
                        included=np.concatenate([pl["included"] for pl in O]),
                        subject=np.concatenate([[pl["subject"]] * len(pl["roi_type"]) for pl in O])))
print("ROIs:", len(roi), "; included:", int(roi.included.sum()))
print(pd.crosstab([roi.subject, roi.type], roi.included).to_string())
Fall = np.concatenate([(pl["F"] - 0.7 * pl["Fnp"])[:, pl["included"]].min(0) for pl in O])
print("Included ROIs whose neuropil-corrected F (r = 0.7) reaches 0 or below:", int((Fall <= 0).sum()))
print("Raw F range (a.u.):", round(min(pl["F"].min() for pl in O), 1), "to", round(max(pl["F"].max() for pl in O), 1))
ratio = np.concatenate([np.median(pl["F"] / np.maximum(pl["Fnp"], 1e-9), 0)[pl["included"]] for pl in O])
print("Included ROIs whose median F / F_neuropil is below 1.05 (Methods: cells at least 5% brighter than the neuropil):",
      int((ratio < 1.05).sum()), "of", len(ratio))
f0bad = 0; f0all = 0
for pl in O:
    X, rel, right, _ = alm.plane_trials(pl, "nonearly")
    f0all += X.shape[0] * int(pl["included"].sum())
    f0bad += int((~np.isfinite(X[:, :, pl["included"]]).all(1)).sum())
print("Included ROI-trials with a non-finite DF/F (F0 of 0):", f0bad, "of", f0all)

In [ ]:
lick = alm.lick_trace_check(assets[assets.kind == "ephys"].asset_id.iloc[0])
print({k: v for k, v in lick.items() if k not in ("mean_trace", "t_ms")})
print("Mean lick trace around the go cue of correct trials (a.u.), 5 ms bins:")
print(pd.Series(lick["mean_trace"].round(4), index=lick["t_ms"]).iloc[30:70].to_string())

The imaging rate is 14.84 Hz (Methods, 15 Hz), and the fluorescence spans every trial. There are
5,215 ROIs, of which 4,706 are marked included, the paper's cell count; the inclusion flag is taken
to stand for the Methods' selection, which the files do not describe (some included ROIs are not
5% brighter than their neuropil by the median ratio; counts above). The neuropil-corrected signal
of most included ROIs touches 0 or below in some frame, which is noise on a small signal; F0 is
averaged over 0.5 s, and the count of ROI-trials where it is 0 is printed above.
Besides PT, IT and unknown, the files label 23 ROIs "Pyr" and 9 "interneuron", labels the paper does
not describe. Both clock checks show selectivity in the response epoch far above the pre-sample
level in almost every session and plane (sessions and planes where it does not are listed above),
so the spike times, the fluorescence and the trial events are consistent with one clock. The lick
trace is stored in 5 s segments at 10 kHz, one per trial from its start, and rises within about
50 to 100 ms after the go cue, consistent with the mouth-movement signal of the Methods; it is used
only in the data check, since the claims file gives no rule for a reaction time (question asked).

## behaviour-performance

> Mice in the object location discrimination task performed at 78.4% correct on average, and fewer
> than 13% of responses came before the go cue.

Criterion: per cent correct excluding lick early trials, on trials without photostimulation; mean
formed (a) over all trials pooled, (b) over sessions, (c) over mice; within 78.4 +/- 7.84%, and the
fraction of lick early trials below 14.3% under the same reading. Trials with tagging light are
left out (the data check finds them from the laser series); the result with them is a variant.
Trials without events are left out (see the data check).
Early licks are flagged only in the extracellular files, so those 99 sessions are the main test and
the imaging sessions a variant. No-response ("ignore") trials are excluded, or counted as errors
(question asked).

In [ ]:
def behaviour(S, kind, light="none"):
    rows = []
    for i, s in enumerate(S):
        tr = s["trials"]
        has = tr["sample"].notna() & tr.instruction.isin(["left", "right"])
        if light == "none" and "light" in tr:
            has &= ~tr.light
        early = tr.early.eq("early")
        rows.append(dict(kind=kind, session=i, subject=s["subject"], trials=int(has.sum()),
                         hit=int((has & ~early & (tr.outcome == "hit")).sum()),
                         miss=int((has & ~early & (tr.outcome == "miss")).sum()),
                         ignore=int((has & ~early & (tr.outcome == "ignore")).sum()),
                         early=int((has & early).sum())))
    return pd.DataFrame(rows)


out = []
early = {}
for light in ("none", "any"):
    B = pd.concat([behaviour(E, "ephys", light), behaviour(O, "ophys", light)], ignore_index=True)
    for kind in ("ephys", "ophys", "both"):
        d = B if kind == "both" else B[B.kind == kind]
        for ign in ("excluded", "as error"):
            den = d.hit + d.miss + (d.ignore if ign == "as error" else 0)
            m = d.assign(den=den).groupby("subject")[["hit", "den"]].sum()
            out.append(dict(light_trials="left out" if light == "none" else "kept", sessions=kind, ignore=ign,
                            n_sessions=len(d), n_mice=d.subject.nunique(), trials=int(d.trials.sum()),
                            a_pooled=100 * d.hit.sum() / den.sum(), b_sessions=100 * (d.hit / den).mean(),
                            c_mice=100 * (m.hit / m.den).mean()))
    d = B[B.kind == "ephys"]
    early[light] = dict(a_pooled=100 * d.early.sum() / d.trials.sum(), b_sessions=100 * (d.early / d.trials).mean(),
                        c_mice=100 * (d.groupby("subject").early.sum() / d.groupby("subject").trials.sum()).mean())
perf = pd.DataFrame(out)
print("Per cent correct (non-early trials with events), readings (a) pooled, (b) mean over sessions, (c) mean over mice:")
print(perf.round(1).to_string(index=False))
print("\nPer cent of trials with an early lick (extracellular sessions), light trials left out:",
      {k: round(v, 1) for k, v in early["none"].items()}, "; kept:", {k: round(v, 1) for k, v in early["any"].items()})
B = pd.concat([behaviour(E, "ephys"), behaviour(O, "ophys")], ignore_index=True)
d = B[B.kind == "ephys"]
print("Paper: 78.4% correct (tolerance 70.56 to 86.24), early responses below 13% (limit 14.3%)")
print("\nPer mouse, extracellular sessions (ignore excluded):")
pm = d.groupby("subject")[["trials", "hit", "miss", "early"]].sum()
pm["pct_correct"] = 100 * pm.hit / (pm.hit + pm.miss)
pm["pct_early"] = 100 * pm.early / pm.trials
print(pm.round(1).to_string())
print("Range of per-session per cent correct:", round((100 * d.hit / (d.hit + d.miss)).min(), 1), "to",
      round((100 * d.hit / (d.hit + d.miss)).max(), 1))

**Result: reproduced.** In the 99 extracellular sessions without the trials with light, per cent
correct is 77.2% pooled, 76.5% over sessions and 75.9% over mice (ignore trials excluded; 76.5%,
75.8% and 75.2% counted as errors), all within 78.4 +/- 7.84%. Early licks are 11.6% of trials over
sessions, below 14.3%, but 14.5% pooled and 14.7% over mice, above it, so the claim holds under
reading (b). Keeping the light trials gives 76.6%, 76.3% and 75.7% correct and 13.0%, 11.7% and
14.7% early licks. The imaging sessions, where early licks are not flagged, give 83.5% pooled. The
dandiset holds 23 of the paper's 42 mice.

## units-classification

> 1,408 single units were recorded from left ALM in 19 mice over 99 sessions; by spike width 124
> were fast-spiking, 1,245 putative pyramidal and 39 intermediate.

Criterion: spike width is the trough-to-peak interval of the mean waveform; fast-spiking below
0.35 ms, putative pyramidal above 0.45 ms, intermediate between; each count within 10%.

In [ ]:
D = U[~U.is_duplicate]
expected = dict(single_units=(1408, 140.8), sessions=(99, 9.9), mice=(19, 1.9), fast_spiking=(124, 12.4),
                putative_pyramidal=(1245, 124.5), intermediate=(39, 3.9))
measured = dict(single_units=len(D), sessions=len(E), mice=len({s["subject"] for s in E}),
                fast_spiking=int((D.width_class == "FS").sum()), putative_pyramidal=int((D.width_class == "Pyr").sum()),
                intermediate=int((D.width_class == "intermediate").sum()))
tab = pd.DataFrame([dict(quantity=k, measured=measured[k], expected=v, tolerance=t,
                         within=abs(measured[k] - v) <= t) for k, (v, t) in expected.items()])
print(tab.to_string(index=False))
print("\nCounting the PT/IT duplicates as units: single units", len(U), ", putative pyramidal by width",
      int((U.width_class == "Pyr").sum()))
print("Units per mouse (distinct):")
print(D.groupby("subject").size().to_string())
plt.figure(figsize=(5, 3))
plt.hist(D.width, bins=np.arange(0.1, 0.75, 1000 / 19531.25) - 0.0256, color="0.4")  # one bin per sample interval
plt.axvline(0.35, color="r"); plt.axvline(0.45, color="r")
plt.xlabel("spike width (ms)"); plt.ylabel("units"); plt.title("trough to peak, distinct units")
plt.tight_layout(); plt.show()
print("Widths plotted:", round(D.width.min(), 3), "to", round(D.width.max(), 3), "ms; none outside the axis")

**Result: partially reproduced.** The dandiset holds 1,369 distinct units (1,441 with the PT/IT
duplicates) in 99 sessions from 19 mice; by spike width 124 are fast-spiking and 1,245 putative
pyramidal, as in the paper. No unit has an intermediate width, against 39 in the paper: five of
the six counts match and the intermediate units are not in the dandiset.

## fig2-selective-fraction

> 73% (912 of 1,245) of putative pyramidal neurons distinguished the trial types in at least one
> trial epoch.

Criterion: per neuron, spike counts in the sample, delay and response epochs compared between lick
right and lick left trials (two-tailed t-test, P < 0.05, uncorrected); selective when significant
in any epoch. Readings: trials (a) correct only or (b) all non-early trials by instruction;
response epoch (i) 1.3 s or (ii) 1.5 s after the go cue.

In [ ]:
P = U[(U.width_class == "Pyr") & ~U.is_duplicate]
EP = {}
rows = []
for light in ("any", "none"):
  for tset, tname in (("correct", "a_correct"), ("nonearly", "b_nonearly")):
    for resp, rname in ((1.3, "i_1.3s"), (1.5, "ii_1.5s")):
        r = alm.epoch_stats(E, P, tset, resp, light)
        EP[(tset, resp, light)] = r
        sig = r[["p_sample", "p_delay", "p_response"]] < 0.05
        sel = sig.any(axis=1)
        prep = sig.p_sample | sig.p_delay
        per = sig.p_response
        rows.append(dict(light_trials="kept" if light == "any" else "left out", trials=tname, response=rname,
                         neurons=len(r), selective=int(sel.sum()),
                         fraction=round(sel.mean(), 3), untestable=int(r[["p_sample", "p_delay", "p_response"]].isna().all(axis=1).sum()),
                         min_trials_right=int(r.n_right.min()), min_trials_left=int(r.n_left.min()),
                         prep_only=int((prep & ~per).sum()), peri_only=int((per & ~prep).sum()), both=int((prep & per).sum())))
SEL = pd.DataFrame(rows)
print(SEL[["light_trials", "trials", "response", "neurons", "selective", "fraction", "untestable", "min_trials_right", "min_trials_left"]].to_string(index=False))
print("Paper: 912 (tolerance 820.8 to 1003.2), fraction 0.73 (0.657 to 0.803)")
SELSET = {light: (EP[("correct", 1.3, light)][["p_sample", "p_delay", "p_response"]] < 0.05).any(axis=1) for light in ("any", "none")}
sel_main = SELSET["any"]
print("\nPer mouse, reading (a)(i):")
print(pd.DataFrame(dict(subject=P.subject, sel=sel_main)).groupby("subject").sel.agg(["size", "sum", "mean"]).round(2).to_string())
print("Chance with three tests at P < 0.05, if independent: about", round(1 - 0.95 ** 3, 3))

**Result: reproduced.** With correct trials and a 1.3 s response epoch, 912 of 1,245 putative
pyramidal neurons (0.733) are selective, the paper's numbers. The other readings give 910 (0.731),
846 (0.680) and 851 (0.684); without the trials with light, 870 (0.699), 868, 795 (0.639) and 792
(0.636). All are within tolerance.

## fig2d-preparatory-perimovement

> Among selective neurons, 219 of 912 had selective preparatory activity only, 389 peri-movement
> activity only, and 304 both.

Criterion: preparatory means significant in the sample or delay epoch, peri-movement significant in
the response epoch, by the test and readings of fig2-selective-fraction; all three counts within
10% under one reading.

In [ ]:
tab = SEL[["light_trials", "trials", "response", "selective", "prep_only", "peri_only", "both"]].copy()
for col, (v, t) in dict(prep_only=(219, 21.9), peri_only=(389, 38.9), both=(304, 30.4)).items():
    tab[col + "_ok"] = (tab[col] - v).abs() <= t
print(tab.to_string(index=False))
print("Paper: preparatory only 219 (197.1 to 240.9), peri-movement only 389 (350.1 to 427.9), both 304 (273.6 to 334.4)")
r = EP[("correct", 1.3, "any")]
sig = r[["p_sample", "p_delay", "p_response"]] < 0.05
print("\nReading (a)(i), light trials kept: significant in sample", int(sig.p_sample.sum()), ", delay", int(sig.p_delay.sum()),
      ", response", int(sig.p_response.sum()), "; delay only (not sample)", int((sig.p_delay & ~sig.p_sample).sum()))

**Result: not reproduced.** Under the main reading, 231 selective neurons are preparatory only
(within tolerance), 284 peri-movement only (paper 389) and 397 both (paper 304). No reading brings
the peri-movement count into tolerance (272 to 294, with or without the light trials); the count of
both is within it in three readings (289 to 331). The split is shifted towards neurons selective in
both epochs.

## fig2c-selectivity-time-course

> Selectivity emerges in the sample epoch, increases through the delay epoch and peaks at the
> beginning of the response epoch.

Criterion: number of putative pyramidal neurons significantly selective in 200 ms windows
(two-tailed t-test, P < 0.05, correct trials), window step 200 ms and 50 ms, both run. Reproduced
when, under each step, (1) some sample-epoch window exceeds the 95th percentile of a binomial count
with p = 0.05 while the pre-sample windows do not exceed it on average; (2) the last delay window
exceeds the first; (3) the maximum falls at the beginning of the response epoch, read as (a) the
first window after the go cue or (b) any window starting within 0.5 s after it. Windows are placed
on a grid with an edge at the sample event; spikes are stored from 0.57 s before it.

In [ ]:
TC = {}
for light, step in [(l, st) for l in ("any", "none") for st in (0.2, 0.05)]:
    first = -0.4 if step == 0.2 else -0.55
    starts = np.round(np.arange(first, 4.9 - 0.2 + 1e-9, step), 3)
    W = alm.window_counts(E, P, starts, light=light)
    nsig = np.zeros(len(starts), int)
    ntest = np.zeros(len(starts), int)
    for R, L in W.values():
        p = stats.ttest_ind(R, L, axis=0).pvalue
        nsig += p < 0.05
        ntest += ~np.isnan(p)
    thr = stats.binom.ppf(0.95, ntest, 0.05)
    TC[(step, light)] = (starts, nsig, thr)
    pre = starts + 0.2 <= 1e-9
    samp = (starts >= -1e-9) & (starts + 0.2 <= 1.3 + 1e-9)
    dl = (starts >= 1.3 - 1e-9) & (starts + 0.2 <= 2.6 + 1e-9)
    resp = starts >= 2.6 - 1e-9
    im = int(np.argmax(nsig))
    c1 = (nsig[samp] > thr[samp]).any() and nsig[pre].mean() <= thr[pre].mean()
    c2 = nsig[dl][-1] > nsig[dl][0]
    c3a = abs(starts[im] - 2.6) < 1e-6
    c3b = (starts[im] >= 2.6 - 1e-9) and (starts[im] < 3.1 - 1e-9)
    print(f"light trials {'kept' if light == 'any' else 'left out'}, step {int(step * 1000)} ms: {len(starts)} windows, neurons tested per window {ntest.min()} to {ntest.max()}, "
          f"binomial 95th percentile {thr.min():.0f} to {thr.max():.0f}")
    print("  pre-sample windows (start, count):", list(zip(starts[pre], nsig[pre])), "mean", round(nsig[pre].mean(), 1))
    print("  sample epoch: max count", nsig[samp].max(), "first window above threshold starts at",
          starts[samp][np.argmax(nsig[samp] > thr[samp])], "s; (1) holds:", c1)
    print("  delay: first window", starts[dl][0], "s count", nsig[dl][0], "; last window", starts[dl][-1], "s count",
          nsig[dl][-1], "; (2) holds:", c2)
    print("  maximum", nsig[im], "in the window starting", round(starts[im] - 2.6, 3), "s after the go cue; first response window",
          nsig[resp][0], "; (3a) holds:", c3a, "; (3b) holds:", c3b)
    print("  five largest windows (start after go cue, count):", [(round(starts[i] - 2.6, 2), int(nsig[i])) for i in np.argsort(-nsig)[:5]])
fig, ax = plt.subplots(figsize=(7, 3))
for (step, light), (starts, nsig, thr) in TC.items():
    ax.plot(starts + 0.1 - 2.6, nsig, label=f"step {int(step * 1000)} ms, light trials {'kept' if light == 'any' else 'left out'}")
ax.plot(TC[(0.05, "any")][0] + 0.1 - 2.6, TC[(0.05, "any")][2], "k:", label="binomial 95th pct")
for x in (-2.6, -1.3, 0):
    ax.axvline(x, color="0.7")
ax.set_xlabel("window centre, s from go cue"); ax.set_ylabel("selective neurons"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("Counts plotted:", min(v[1].min() for v in TC.values()), "to", max(v[1].max() for v in TC.values()))

**Result: reproduced.** Under both steps, pre-sample windows hold 52 to 65 selective neurons
against a chance limit of 71 to 74, sample-epoch windows reach 272 (200 ms step) and 280 (50 ms
step), selectivity grows through the delay (326 to 408 and 321 to 408), and the maximum (574 and
582 neurons) lies 0.2 s and 0.35 s after the go cue. The first window after the go cue holds 500,
so reading (3a) fails and reading (3b) holds under both steps. Without the trials with light the
same holds (pre-sample 47 to 67, sample maximum 248 and 249, delay 273 to 377 and 281 to 377,
maximum 525 and 538 at 0.2 s and 0.25 s, first response window 461).

## fig2c-population-dynamics

> Individual neuron responses are dynamic although selectivity is stable at the level of the
> population (Fig. 2c, bottom, population selectivity and the correlation of the population
> response with that at the go cue).

Criterion: pseudo-population of (a) the 912 selective or (b) all 1,245 putative pyramidal neurons;
each neuron's response (correct trials, 200 ms bins) mean-subtracted and normalized to its variance
over the trial; Pearson correlation of the population vector at each time with that at the go cue,
the vector concatenating both trial types or taking their difference; population selectivity as
preferred minus non-preferred rate, the preference set from 10 trials and computed on the rest.
Reproduced when, under each reading, the mean correlation over the sample epoch is below that over
the delay epoch, which is below 1, and the population selectivity stays above 0 through the delay.
Details the criterion leaves open (question asked) are all run: division by the variance or the
standard deviation, 10 trials in all or 10 of each type, and the go-cue bin starting or ending at
the go cue.

In [ ]:
rng = np.random.default_rng(2024)
starts = np.round(np.arange(-3.0, 2.0 + 1e-9, 0.2), 3)  # bin starts relative to the go cue
nb = len(starts)
samp = (starts >= -2.6 - 1e-9) & (starts + 0.2 <= -1.3 + 1e-9)
dl = (starts >= -1.3 - 1e-9) & (starts + 0.2 <= 1e-9)
rows = []
curves = {}
for light in ("any", "none"):
    lname = "kept" if light == "any" else "left out"
    Wg = alm.window_counts(E, P, starts, align="go", light=light)
    sel_idx = set(P.index[SELSET[light].reindex(P.index).values])
    Rm = {k: R.mean(0) / 0.2 for k, (R, L) in Wg.items()}
    Lm = {k: L.mean(0) / 0.2 for k, (R, L) in Wg.items()}
    popsel = {}
    for mode in ("10 in all", "10 per type"):
        vals = {}
        for k, (R, L) in Wg.items():
            n_r, n_l = len(R), len(L)
            if mode == "10 in all":
                pick = rng.choice(n_r + n_l, 10, replace=False)
                pr, pl = pick[pick < n_r], pick[pick >= n_r] - n_r
            else:
                pr, pl = rng.choice(n_r, min(10, n_r), replace=False), rng.choice(n_l, min(10, n_l), replace=False)
            restR, restL = np.setdiff1d(np.arange(n_r), pr), np.setdiff1d(np.arange(n_l), pl)
            if len(restR) == 0 or len(restL) == 0:
                continue
            mr = R[pr].sum(1).mean() if len(pr) else 0.0
            ml = L[pl].sum(1).mean() if len(pl) else 0.0
            sign = 1 if mr >= ml else -1
            vals[k] = sign * (R[restR].mean(0) - L[restL].mean(0)) / 0.2
        popsel[mode] = vals
    for pop in ("a_selective", "b_all"):
        keys = [k for k in Wg if (k in sel_idx or pop == "b_all")]
        X = np.array([np.concatenate([Rm[k], Lm[k]]) for k in keys])
        for norm in ("variance", "sd"):
            scale = X.var(1) if norm == "variance" else X.std(1)
            keep = scale > 0
            Z = (X[keep] - X[keep].mean(1, keepdims=True)) / scale[keep][:, None]
            Zr, Zl = Z[:, :nb], Z[:, nb:]
            for vec in ("concatenated", "difference"):
                V = np.concatenate([Zr, Zl], 0) if vec == "concatenated" else Zr - Zl
                for ref_name, ref in (("bin starting at go", int(np.argmin(abs(starts)))), ("bin ending at go", int(np.argmin(abs(starts + 0.2))))):
                    c = np.array([np.corrcoef(V[:, i], V[:, ref])[0, 1] for i in range(nb)])
                    curves[(light, pop, norm, vec, ref_name)] = c
                    rows.append(dict(light_trials=lname, neurons=pop, n=int(keep.sum()), norm=norm, vector=vec, reference=ref_name,
                                     r_sample=c[samp].mean(), r_delay=c[dl].mean(), holds=c[samp].mean() < c[dl].mean() < 1))
        for mode, vals in popsel.items():
            ps = np.array([vals[k] for k in keys if k in vals])
            boot = np.array([ps[rng.integers(0, len(ps), len(ps))].mean(0) for _ in range(1000)])
            lo = np.percentile(boot, 2.5, axis=0)
            rows.append(dict(light_trials=lname, neurons=pop, n=len(ps), norm="", vector=f"population selectivity, preference from {mode}",
                             reference="", r_sample=ps.mean(0)[samp].mean(), r_delay=ps.mean(0)[dl].min(),
                             holds=bool((ps.mean(0)[dl] > 0).all())))
            print(f"light trials {lname}, {pop}, preference from {mode}: population selectivity per delay bin (spikes/s)",
                  ps.mean(0)[dl].round(2).tolist(), "; bootstrap 2.5th percentile", lo[dl].round(2).tolist(),
                  f"; neurons without remaining trials {len(keys) - len(ps)}")
res_pd = pd.DataFrame(rows)
print("\nCorrelation rows: r_sample and r_delay are the mean correlations over the sample and delay bins.")
print("Selectivity rows: r_sample is the mean over sample bins, r_delay the minimum over delay bins (spikes/s).")
print(res_pd.round(3).to_string(index=False))
print("Readings that hold:", int(res_pd.holds.sum()), "of", len(res_pd))
fig, ax = plt.subplots(figsize=(7, 3))
for key in [("any", "a_selective", "variance", "concatenated", "bin starting at go"), ("any", "a_selective", "sd", "difference", "bin ending at go")]:
    ax.plot(starts + 0.1, curves[key], label=", ".join(key[2:]))
for x in (-2.6, -1.3, 0):
    ax.axvline(x, color="0.7")
ax.set_xlabel("bin centre, s from go cue"); ax.set_ylabel("correlation with go-cue vector"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("Plotted correlations range:", round(min(curves[k].min() for k in curves), 3), "to", round(max(curves[k].max() for k in curves), 3))

**Result: blocked (methods underspecified).** 39 of the 40 rows hold. With the light trials kept,
under all 16 correlation readings the mean correlation with the go-cue vector is lower over the
sample epoch (-0.12 to 0.24) than over the delay epoch (0.04 to 0.78), the delay value is below 1,
and the population selectivity stays above 0 in every delay bin (minimum 0.62 spikes/s). Without the
light trials, one reading fails: all neurons, divided by the variance, difference vector, bin
starting at the go cue (sample 0.078 against delay 0.054). The criterion asks the claim to hold
under each of its readings, so the outcome turns on details it leaves open (whether light trials
are kept, the normalization, the go-cue bin; questions asked).

## ed2c-error-trials-predict-choice

> On error trials ALM activity predicts the direction the mouse licked, so most selective neurons
> switch their trial-type preference relative to correct trials.

Criterion: for the 912 selective neurons and each epoch, selectivity (rate lick right minus rate
lick left, by instructed trial type) on correct and on error trials; Pearson correlation across
neurons. Readings: at least (a) one or (b) five error trials of each type. Reproduced when r < 0 in
the delay and response epochs under each reading.

In [ ]:
rows = []
for light in ("any", "none"):
    selU = P[SELSET[light].reindex(P.index).values]
    c_st = alm.epoch_stats(E, selU, "correct", 1.3, light)
    e_st = alm.epoch_stats(E, selU, "error", 1.3, light)
    for mn in (1, 5):
        ok = (e_st.n_right >= mn) & (e_st.n_left >= mn)
        for ep in ("sample", "delay", "response"):
            x = (c_st["R_" + ep] - c_st["L_" + ep])[ok]
            y = (e_st["R_" + ep] - e_st["L_" + ep])[ok]
            r, p = stats.pearsonr(x, y)
            rows.append(dict(light_trials="kept" if light == "any" else "left out", min_error_trials=mn, epoch=ep,
                             neurons=int(ok.sum()), left_out=len(selU) - int(ok.sum()), r=r, p=p,
                             switched=np.mean(np.sign(x) != np.sign(y))))
    if light == "any":
        ok = (e_st.n_right >= 1) & (e_st.n_left >= 1)
        x = (c_st.R_delay - c_st.L_delay)[ok]; y = (e_st.R_delay - e_st.L_delay)[ok]
        per_mouse = pd.DataFrame(dict(subject=selU.subject[ok], x=x, y=y)).groupby("subject").apply(
            lambda g: pd.Series(dict(n=len(g), r=stats.pearsonr(g.x, g.y)[0] if len(g) > 2 else np.nan)))
ED2 = pd.DataFrame(rows)
print(ED2.round(4).assign(p=[f"{x:.1e}" for x in ED2.p]).to_string(index=False))
print("Per mouse, delay epoch, at least one error trial of each type, light trials kept:")
print(per_mouse.round(2).to_string())

**Result: reproduced.** Selectivity on error trials is anticorrelated with that on correct trials
in the delay epoch (r = -0.30 with at least one error trial of each type, 897 neurons; -0.25 with
at least five, 833 neurons) and the response epoch (-0.38 and -0.34), all P < 1e-12. Without the
light trials, r is -0.29 and -0.32 (delay) and -0.33 and -0.30 (response). In the sample epoch the
correlation is near 0 (-0.025 to 0.045).

## fig2e-no-contra-bias-spikes

> Contra-preferring and ipsi-preferring neurons are present in equal proportions in left ALM, and
> the overall spike count shows no significant contralateral bias (P = 0.13; 8% spike rate
> modulation across all task-selective neurons).

Criterion: a claim of absence. Two-tailed t-test against 0 of the spike count difference (lick
right minus lick left, correct trials) across neurons; reproduced when P >= 0.05. The criterion
names readings: neurons (a) all 1,245 or (b) the 912 selective; window (i) the three epochs
together, (ii) the 570 ms before the go cue, (iii) the delay epoch. The protocol judges absence on
the paper's test as written, so the outcome rests on (a)(i), "the overall spike count across the
recorded population"; the others are shown. The proportions are tested with the bootstrap of
Fig. 3f, j, which must not find a contralateral excess at P < 0.05. The 8% modulation is checked
with the readings of fig3f-pt-contra-selectivity.

In [ ]:
rng = np.random.default_rng(7)
rows, boots = [], []
for light in ("any", "none"):
    lname = "kept" if light == "any" else "left out"
    selU = P[SELSET[light].reindex(P.index).values]
    for pop, units in (("a_all", P), ("b_selective", selU)):
        for win, (t0, t1) in (("i_three_epochs", (None, None)), ("ii_570ms", (-0.57, 0.0)), ("iii_delay", (-1.3, 0.0))):
            if t0 is None:
                st = alm.epoch_stats(E, units, "correct", 1.3, light)
                Rr = sum(st["R_" + e] * 1.3 for e in ("sample", "delay", "response"))
                Ll = sum(st["L_" + e] * 1.3 for e in ("sample", "delay", "response"))
            else:
                w = alm.window_diff(E, units, t0, t1, light=light)
                Rr, Ll = w.R * (t1 - t0), w.L * (t1 - t0)
            diff = Rr - Ll
            t = stats.ttest_1samp(diff, 0)
            rows.append(dict(light_trials=lname, neurons=pop, window=win, n=len(diff), mean_count_diff=diff.mean(), t=t.statistic,
                             P=t.pvalue, mod_x_pct=100 * diff.mean() / ((Rr + Ll) / 2).mean(), mod_y_pct=100 * diff.mean() / Ll.mean()))
    contra = (alm.total_count_diff(E, selU, light=light) > 0).values
    pb = alm.boot_more_ipsi(contra, rng)
    boots.append(dict(light_trials=lname, selective=len(contra), contra=int(contra.sum()), fraction=round(contra.mean(), 3),
                      P_more_ipsi=pb, P_contra_at_least_ipsi=1 - pb, binomial_P=stats.binomtest(int(contra.sum()), len(contra), 0.5).pvalue))
    if light == "any":
        by_mouse = pd.DataFrame(dict(subject=selU.subject.values, contra=contra)).groupby("subject").contra.agg(["sum", "size"])
FE = pd.DataFrame(rows)
print(FE.round(4).to_string(index=False))
print("Paper: P = 0.13; modulation 8% (tolerance 7.2 to 8.8)")
print("\nContra- and ipsi-preferring selective neurons, bootstrap (neurons resampled, 10^4):")
print(pd.DataFrame(boots).round(4).to_string(index=False))
print("By mouse, light trials kept (contra-preferring / selective):")
print(by_mouse.to_string())

**Result: reproduced.** On the paper's test as written, the overall spike count difference across
the 1,245 putative pyramidal neurons is not different from 0 (P = 0.82, or 0.52 without the light
trials; 0.67 and 0.35 for the selective neurons). The bootstrap finds no contralateral excess: 419
of 912 selective neurons are contra-preferring (0.459; 408 of 870 without light trials). The
proportions are not equal, though: ipsi-preferring neurons are the majority (bootstrap P = 0.99 and
0.97 for more ipsi than contra; two-sided binomial P = 0.016 and 0.072). In the 570 ms before the go
cue the difference reaches P = 0.022 to 0.050 across neurons and light readings, and the delay
epoch gives P = 0.051 to 0.12. The 8% modulation is matched in some readings (all neurons,
570 ms window, difference over the mean rate: 8.5%; selective neurons, delay, light trials left
out: 8.0% and 8.3%), not in others (5.8% to 10.7%).

## ed1-fast-spiking-properties

> Fast-spiking neurons have higher baseline firing rates and are less selective than putative
> pyramidal neurons.

Criterion: baseline (a) the 0.5 s before pole movement onset or (b) all recorded time before it in
each trial; selectivity (i) the fraction significantly selective by the test of
fig2-selective-fraction or (ii) the mean absolute spike rate difference between trial types over
the three epochs. Reproduced when, under every reading, the fast-spiking median and mean baseline
exceed the pyramidal ones and their selectivity is lower; a two-sided Mann-Whitney test is
reported.

In [ ]:
FSu = U[(U.width_class == "FS") & ~U.is_duplicate]
def props(units, light):
    st = alm.epoch_stats(E, units, "correct", 1.3, light)
    rows = []
    for u in units.itertuples():
        s = E[u.session]
        tr = s["trials"]
        t = tr[alm.trial_sets(tr, light)["correct"]]
        sp = s["spikes"][u.unit]
        rows.append(dict(idx=u.Index,
                         base_a=(alm.counts(sp, t["sample"].values - 0.5, t["sample"].values) / 0.5).mean(),
                         base_b=(alm.counts(sp, t.start.values, t["sample"].values) / (t["sample"] - t.start).values).mean()))
    d = pd.DataFrame(rows).set_index("idx")
    d["selective"] = (st[["p_sample", "p_delay", "p_response"]] < 0.05).any(axis=1)
    d["absdiff"] = np.mean([abs(st["R_" + e] - st["L_" + e]) for e in ("sample", "delay", "response")], axis=0)
    return d
rows = []
for light in ("any", "none"):
    fs, py = props(FSu, light), props(P, light)
    for k in ("base_a", "base_b", "absdiff"):
        rows.append(dict(light_trials="kept" if light == "any" else "left out", measure=k, FS_median=fs[k].median(), FS_mean=fs[k].mean(),
                         Pyr_median=py[k].median(), Pyr_mean=py[k].mean(), MW_P=f"{stats.mannwhitneyu(fs[k], py[k]).pvalue:.1e}"))
    rows.append(dict(light_trials="kept" if light == "any" else "left out", measure="fraction selective", FS_median=np.nan,
                     FS_mean=fs.selective.mean(), Pyr_median=np.nan, Pyr_mean=py.selective.mean(),
                     MW_P=f"Fisher {stats.fisher_exact([[fs.selective.sum(), (~fs.selective).sum()], [py.selective.sum(), (~py.selective).sum()]])[1]:.3f}"))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("For 'fraction selective', FS_mean and Pyr_mean are the fractions; FS units:", len(FSu), "; Pyr units:", len(P))
print("Pre-sample time recorded per trial (s), the span of baseline (b):",
      round(float(np.median(np.concatenate([(s['trials']['sample'] - s['trials'].start).dropna().values for s in E]))), 3))
print("FS units per mouse:", FSu.groupby("subject").size().to_dict())

**Result: partially reproduced.** Fast-spiking neurons have higher baseline rates under both
readings (median 4.51 against 1.50 spikes/s and mean 6.90 against 2.92 over the 0.5 s before the
sample; Mann-Whitney P = 1e-17). They are not less selective: 0.750 of them are selective against
0.733 of putative pyramidal neurons (0.710 and 0.699 without the light trials), and their mean
absolute rate difference is larger (median 1.46 against 0.62 spikes/s). The baseline part holds, the selectivity part does not, under these
unnormalized measures.

## tagging-counts

> Of 73 neurons antidromically activated from pyramidal tract axons, 45 passed the collision test
> (PT neurons), 22 failed and 6 could not be tested; of 134 activated from IT axons, 27 passed (IT
> neurons), 106 failed and 1 could not be tested.

Criterion: counts of units by the labels the dataset gives for tagging outcome.

In [ ]:
lab = U.label.value_counts()
tab = pd.DataFrame([dict(quantity="PT neurons", measured=int(lab.get("PT", 0)), expected=45, tolerance=4.5),
                    dict(quantity="IT neurons", measured=int(lab.get("IT", 0)), expected=27, tolerance=2.7),
                    dict(quantity="PT-activated, failed collision", measured=None, expected=22, tolerance=2.2),
                    dict(quantity="IT-activated, failed collision", measured=None, expected=106, tolerance=10.6)])
tab["within"] = [abs(m - e) <= t if m is not None else None for m, e, t in zip(tab.measured, tab.expected, tab.tolerance)]
print(tab.to_string(index=False))
print("Unit labels in the files:", sorted(U.label.unique()), "; no label marks activated neurons that failed the collision test or could not be tested")
print("Tagged units by mouse and genotype:")
g = pd.DataFrame(dict(subject=U.subject, label=U.label, genotype=[E[s]["genotype"] for s in U.session]))
print(g[g.label.isin(["PT", "IT"])].groupby(["genotype", "subject", "label"]).size().to_string())

**Result: partially reproduced.** The files label 45 PT and 27 IT units, the paper's counts. The
22 and 106 activated neurons that failed the collision test, and the 7 that could not be tested,
carry no label, so half of the counts cannot be checked.

## antidromic-latencies (blocked; variant shown)

> PT neurons had shorter antidromic latencies (mean 2.7 ms) than IT neurons (4.6 ms), and both had
> smaller jitter (0.1 and 0.2 ms) than activated neurons that failed collision (latency 8.3 and
> 9.8 ms, jitter 2.8 and 2.4 ms).

Criterion: latency is the time from light pulse onset to the antidromic spike, jitter its s.d. over
pulses, per neuron, then averaged over neurons; tested only if the dataset holds the tagging
trials. It does (data check), but the criterion does not say how the antidromic spike is told
apart from spontaneous spikes (a window, a response criterion), and the jitter depends on it
(question asked). The claim is blocked as methods_underspecified. As a variant that decides
nothing, the first spike within 10 ms after the first pulse of each pair (the gap to the second
pulse) is shown for every PT and IT unit, with the probability of a spike in the 10 ms before
the pulse as a reference; PT units are aligned to the pons light and IT units to the ALM light. The activated neurons that failed the collision test carry no label.

In [ ]:
rows = []
for u in U[U.label.isin(["PT", "IT"])].itertuples():
    s = E[u.session]
    site = "PONS" if u.label == "PT" else "ALM"  # PT neurons are tagged from the pons, IT neurons from the other ALM
    series = [v["onsets"] for k, v in s["laser"].items() if site in k]
    on = np.concatenate(series) if series else np.array([])
    row = dict(label=u.label, subject=u.subject, genotype=s["genotype"], pair_onsets=0)
    if len(on):
        first = alm.pair_onsets(on)
        sp = s["spikes"][u.unit]
        lat = alm.first_spike_latency(sp, first, 0.010) * 1000
        pre = alm.first_spike_latency(sp, first - 0.010, 0.010)
        l = lat[~np.isnan(lat)]
        row.update(pair_onsets=len(first), p_spike_after=np.mean(~np.isnan(lat)), p_spike_before=np.mean(~np.isnan(pre)),
                   median_latency_ms=np.median(l) if len(l) else np.nan, sd_latency_ms=np.std(l) if len(l) > 1 else np.nan)
    rows.append(row)
AL = pd.DataFrame(rows)
print(AL.round(3).to_string(index=False))
print("\nUnits without a light series of their site in their session:", int((AL.pair_onsets == 0).sum()),
      AL[AL.pair_onsets == 0].groupby(["genotype", "label"]).size().to_dict())
g = AL[AL.pair_onsets > 0].groupby("label")[["p_spike_after", "p_spike_before", "median_latency_ms", "sd_latency_ms"]]
print("Means over units (variant, 10 ms window after the first pulse of a pair):")
print(g.mean().round(3).to_string())
print("Units per label with a light series:", AL[AL.pair_onsets > 0].label.value_counts().to_dict())
print("Paper: PT 2.7 ms latency, 0.1 ms jitter; IT 4.6 ms, 0.2 ms")

## fig3e-pt-it-both-activity-types

> Both PT and IT neurons show preparatory and movement-related activity.

Criterion: the 45 PT and 27 IT neurons classified with the epoch rule and readings of
fig2d-preparatory-perimovement; reproduced when, in each cell type, at least one neuron has
preparatory selectivity and at least one peri-movement selectivity.

In [ ]:
rows = []
for (tset, resp, light), r in EP.items():
    sig = r[["p_sample", "p_delay", "p_response"]] < 0.05
    for tag in ("PT", "IT"):
        s = sig.loc[P.index[P.tag == tag]]
        prep, per = s.p_sample | s.p_delay, s.p_response
        rows.append(dict(light_trials="kept" if light == "any" else "left out", trials=tset, response=resp, type=tag,
                         n=len(s), selective=int((prep | per).sum()),
                         preparatory=int(prep.sum()), peri_movement=int(per.sum()), prep_only=int((prep & ~per).sum()),
                         peri_only=int((per & ~prep).sum()), both=int((prep & per).sum())))
F3E = pd.DataFrame(rows)
print(F3E.to_string(index=False))
print("Holds under every reading:", bool(((F3E.preparatory > 0) & (F3E.peri_movement > 0)).all()))

**Result: reproduced.** Under the main reading, 17 of 45 PT neurons have preparatory and 24
peri-movement selectivity, and 13 of 27 IT neurons each; every reading, with or without the light
trials, gives at least 11 neurons of each kind in each cell type.

## fig3f-pt-contra-preferring

> Contra-preferring neurons outnumber ipsi-preferring neurons among PT neurons (P < 0.05,
> bootstrap) but not among IT neurons (P = 0.74).

Criterion: contra- or ipsi-preferring by total spike count over the three epochs, lick right
against lick left, correct trials; bootstrap with neurons resampled (10^4), P the fraction of
resamples with more ipsi- than contra-preferring neurons. Readings: (a) all tagged neurons of the
type, (b) only those selective by the test of fig2-selective-fraction. Reproduced when PT gives
P < 0.05 and IT P >= 0.05 under the same reading.

In [ ]:
rng = np.random.default_rng(31)
rows = []
for light in ("any", "none"):
    tot_all = alm.total_count_diff(E, P[P.tag != ""], light=light)
    for reading in ("a_all", "b_selective"):
        for tag in ("PT", "IT"):
            idx = P.index[(P.tag == tag) & (SELSET[light].reindex(P.index).values if reading == "b_selective" else True)]
            x = (tot_all.loc[idx] > 0).values
            rows.append(dict(light_trials="kept" if light == "any" else "left out", reading=reading, type=tag, n=len(x),
                             contra=int(x.sum()), fraction=round(x.mean(), 3), P_boot=alm.boot_more_ipsi(x, rng)))
    if light == "any":
        by_mouse = pd.DataFrame(dict(subject=P.subject[P.tag == "PT"], c=(tot_all.loc[P.index[P.tag == "PT"]] > 0).values)).groupby("subject").c.agg(["sum", "size"])
F3F = pd.DataFrame(rows)
print(F3F.to_string(index=False))
print("Paper: PT P < 0.05, IT P = 0.74")
print("PT neurons by mouse, light trials kept (contra / n):")
print(by_mouse.to_string())

**Result: blocked (methods underspecified).** With the light trials kept, 30 of 45 PT neurons are
contra-preferring (0.67; bootstrap P = 0.010) and 13 of 27 IT neurons (0.48; P = 0.58), so the claim
holds under reading (a); restricted to selective neurons, PT gives 18 of 29 (P = 0.09). Without the
trials with light, PT gives 26 of 45 (0.58; P = 0.14) and 17 of 32 selective (P = 0.30), and the
claim holds under no reading. The outcome turns on whether those trials are kept, which the claims
file does not say (question asked).

## fig3f-pt-contra-selectivity

> Contralateral population selectivity emerges in PT neurons from 570 ms before the response epoch
> (34% spike rate modulation, P < 0.05), but not in IT neurons (-19%, P = 0.40).

Criterion: contra-selectivity is the rate difference, lick right minus lick left, per neuron,
correct trials, averaged across neurons, in 200 ms windows. Readings: neurons (a) all tagged or
(b) selective; test window (i) the 570 ms before the go cue or (ii) the delay epoch; modulation
(x) the difference over the mean rate of both trial types or (y) over the lick left rate; onset the
start of the earliest window from which the population contra-selectivity stays significantly
above 0 (t-test across neurons, P < 0.05) until the go cue, windows stepped by 10 ms, each window
standing for its (p) start, (q) centre or (r) end. Reproduced when, under one reading, PT gives
P < 0.05 with positive selectivity, IT P >= 0.05, and the three values are within tolerance.

In [ ]:
starts = np.round(np.arange(-3.0, 0.0 + 1e-9, 0.01), 3)
rows, onsets, curves3 = [], [], {}
for light in ("any", "none"):
    lname = "kept" if light == "any" else "left out"
    for reading in ("a_all", "b_selective"):
        for tag in ("PT", "IT"):
            units = P[(P.tag == tag) & (SELSET[light].reindex(P.index).values if reading == "b_selective" else True)]
            W = alm.window_counts(E, units, starts, align="go", light=light)
            Dm = np.array([R.mean(0) / 0.2 - L.mean(0) / 0.2 for R, L in W.values()])
            p = stats.ttest_1samp(Dm, 0, axis=0).pvalue
            sigpos = (p < 0.05) & (Dm.mean(0) > 0)
            curves3[(lname, reading, tag)] = Dm.mean(0)
            for label, off in (("p_start", 0.0), ("q_centre", 0.1), ("r_end", 0.2)):
                tt = starts + off
                v = tt <= 1e-9
                sv, tv = sigpos[v], tt[v]
                if sv[-1]:
                    k = len(sv) - 1
                    while k > 0 and sv[k - 1]:
                        k -= 1
                    onset = -tv[k] * 1000
                else:
                    onset = np.nan
                onsets.append(dict(light_trials=lname, reading=reading, type=tag, window_time=label, onset_ms_before_go=onset,
                                   windows_significant_positive=int(sv.sum()), smallest_window_P=float(p[v].min())))
            for win, (t0, t1) in (("i_570ms", (-0.57, 0.0)), ("ii_delay", (-1.3, 0.0))):
                w = alm.window_diff(E, units, t0, t1, light=light)
                d = w.R - w.L
                t = stats.ttest_1samp(d, 0)
                rows.append(dict(light_trials=lname, reading=reading, type=tag, window=win, n=len(d), diff_spikes_s=d.mean(), P=t.pvalue,
                                 mod_x_pct=100 * d.mean() / ((w.R + w.L) / 2).mean(), mod_y_pct=100 * d.mean() / w.L.mean()))
F3S = pd.DataFrame(rows)
print(F3S.round(4).to_string(index=False))
print("\nOnsets (ms before the go cue; NaN where the last window before the go cue is not significant):")
print(pd.DataFrame(onsets).round(4).to_string(index=False))
print("Paper: PT onset 570 ms (513 to 627), PT modulation 34% (30.6 to 37.4), IT -19% (-20.9 to -17.1); PT P < 0.05, IT P = 0.40")
fig, ax = plt.subplots(figsize=(7, 3))
for key, c in curves3.items():
    if key[0] == "kept":
        ax.plot(starts + 0.1, c, label=", ".join(key[1:]))
ax.axhline(0, color="0.7"); ax.axvline(-1.3, color="0.7")
ax.set_xlabel("window centre, s from go cue"); ax.set_ylabel("contra-selectivity (spikes/s)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("Plotted range (light trials kept):", round(min(c.min() for k, c in curves3.items() if k[0] == "kept"), 2), "to",
      round(max(c.max() for k, c in curves3.items() if k[0] == "kept"), 2), "spikes/s")

**Result: not reproduced.** PT neurons have a positive contra-selectivity before the go cue, but the
test-window t-test is not significant under any reading (P = 0.17 to 0.36 with the light trials,
0.12 to 0.22 without), and the windowed selectivity is not significant up to the go cue, so no onset
can be found (without light trials 8 to 16 single windows reach P < 0.05, not continuously). The PT
modulation is 13.9% to 27.0% with the light trials and 18.9% to 38.5% without, against 34%. IT
neurons are not significant (P = 0.06 to 0.13), with modulations of -15.9% to -29.7%.

## imaging-selectivity-counts

> In two-photon imaging of ALM (59 fields of view, 4 mice), 58% of cells (2,740 of 4,706) were
> task-related and 57% of those (1,575) trial-type selective, with a slight majority of
> contra-preferring cells (61%, 971 of 1,575).

Criterion: task-related when a Kruskal-Wallis test across 5-frame bins rejects equal fluorescence
at P < 0.01 in contra or in ipsi trials; selective when a Wilcoxon rank sum test of DF/F between
contra and ipsi trials gives P < 0.05; preference by the average signal over the trial epochs;
DF/F0 with F0 the mean over 0.5 s before trial start, neuropil factor 0.7. Readings: Kruskal-Wallis
span (a) sample onset to the end of the response epoch or (b) the whole recorded trial; rank sum of
(i) the mean over the three epochs or (ii) each epoch; trials correct only or all non-early. The
response epoch is 1.3 s or 1.5 s (question asked); both are run. Cells are the ROIs marked
included; contra trials are lick right trials (left ALM).

In [ ]:
PT_ = {tset: [alm.plane_trials(pl, tset) for pl in O] for tset in ("correct", "nonearly")}
IMG = {}
rows = []
for resp in (1.3, 1.5):
    for span in ("a", "b"):
        for rank in ("i_mean", "ii_each"):
            for tset in ("correct", "nonearly"):
                res = [alm.classify_plane(X, rel, right, pl["included"], resp, span, rank.split("_")[1])
                       for (X, rel, right, _), pl in zip(PT_[tset], O)]
                IMG[(resp, span, rank, tset)] = res
                S = np.concatenate([r["sel"] for r in res]); C = np.concatenate([r["contra"] for r in res])
                rows.append(dict(response=resp, span=span, rank=rank, trials=tset,
                                 cells=int(sum(r["included"].sum() for r in res)), task=int(sum(r["task"].sum() for r in res)),
                                 selective=int(S.sum()), contra_frac=round((S & C).sum() / S.sum(), 3), planes=len(res)))
IM = pd.DataFrame(rows)
for col, (v, t) in dict(cells=(4706, 470.6), task=(2740, 274), selective=(1575, 157.5), contra_frac=(0.61, 0.061), planes=(59, 5.9)).items():
    IM[col + "_ok"] = (IM[col] - v).abs() <= t + 1e-9
IM["all_ok"] = IM[[c for c in IM.columns if c.endswith("_ok")]].all(axis=1)
print(IM.to_string(index=False))
print("Paper: cells 4706, task-related 2740 (2466 to 3014), selective 1575 (1417.5 to 1732.5), contra 0.61 (0.549 to 0.671), planes 59")
print("Readings with every value within tolerance:", int(IM.all_ok.sum()), "of", len(IM))
print("Trials per plane (correct):", pd.Series([len(r) for _, _, r, _ in PT_["correct"]]).describe().round(1).to_dict())
key = (1.5, "a", "i_mean", "nonearly")
print("\nBy mouse, reading", key, "(task-related, selective, included):")
subj = np.concatenate([[pl["subject"]] * len(pl["roi_type"]) for pl in O])
r = IMG[key]
print(pd.DataFrame(dict(subject=subj, task=np.concatenate([x["task"] for x in r]), sel=np.concatenate([x["sel"] for x in r]),
                        inc=np.concatenate([x["included"] for x in r]))).groupby("subject").sum().to_string())

**Result: partially reproduced.** The dandiset holds the paper's 4,706 cells in 59 planes. Several
readings put the task-related count within tolerance (2,551 to 2,772 with span a; 3,104 and 3,134
with span b are above it), and all readings but one put the contra-preferring fraction within it
(0.547 to 0.590, against a lower limit of 0.549). The selective count falls short of 1,575 in every reading where the task-related count fits:
the closest is 1,417 (1.5 s response epoch, span a, mean over epochs, all non-early trials),
just below the lower limit of 1,417.5. No reading puts all five values within tolerance.

## fig4f-spatially-intermingled

> Contra- and ipsi-preferring neurons are found in all fields of view and are spatially
> intermingled; nearest-neighbour selectivity and same-type distances do not differ from shuffled
> data (P = 0.34 and P = 0.61).

Criterion: a claim of absence on the paper's two tests: (1) the probability that each selective
cell's nearest neighbour (among (a) selective cells or (b) all cells) has the same preference,
against shuffled labels, P the fraction of shuffles with a larger probability; (2) the mean distance
to the nearest cell of the same preference, P the fraction of shuffles with a smaller distance;
labels shuffled within each field of view or across all; 10^4 shuffles. Reproduced when both tests
give P >= 0.05 and both preferences occur in every field of view; not reproduced when either test
gives P < 0.05. Distances are between ROI centroids in pixels (the files give no scale). Run for
every reading of imaging-selectivity-counts.

In [ ]:
rng = np.random.default_rng(4)
rows = []
for key, res in IMG.items():
    both = sum(bool((r["sel"] & r["contra"]).any() and (r["sel"] & ~r["contra"]).any()) for r in res)
    for nbset in ("selective", "all"):
        for shuf in ("within", "across"):
            out = alm.spatial_tests(O, res, nbset, shuf, rng)
            rows.append(dict(response=key[0], span=key[1], rank=key[2], trials=key[3], neighbours=nbset, shuffle=shuf,
                             planes_with_both=both, P_prob=out["P_prob"], prob=out["prob"], prob_shuffle=out["prob_shuffle"],
                             P_dist=out["P_dist"], dist=out["dist"], dist_shuffle=out["dist_shuffle"]))
SP = pd.DataFrame(rows)
SP["absent"] = (SP.P_prob >= 0.05) & (SP.P_dist >= 0.05)
pd.set_option("display.max_rows", 100)
print(SP.round(4).to_string(index=False))
print("\nTest rows with both P >= 0.05:", int(SP.absent.sum()), "of", len(SP),
      "; selection readings with both preferences in all 59 planes:", int((SP.groupby(["response", "span", "rank", "trials"]).planes_with_both.first() == 59).sum()), "of 16")
print("Rows with P < 0.05 in at least one test, by neighbour set and shuffle:")
print((~SP.absent).groupby([SP.neighbours, SP.shuffle]).sum().to_string())

**Result: not reproduced.** In 57 of the 64 combinations of selection reading, neighbour set and
shuffle, at least one test finds clustering at P < 0.05 (nearby cells share their preference more
often, and same-preference cells are closer, than with shuffled labels); every combination with
all cells as neighbours does. In 9 of the 16 selection readings, some plane lacks one of the two
preferences. The effect is small (for example a
same-preference probability of 0.539 against 0.517 for shuffles), and neighbouring ROIs share
neuropil, which may contribute.

## fig4h-pt-contra-imaging

> Among layer 5 neurons, about a quarter of labelled PT (182 of 534) and IT (174 of 542) neurons
> were selective; 67% of selective PT neurons were contra-preferring (P < 0.001, binomial test) but
> only 48% of IT neurons, and PT neurons were more often contra-preferring than IT neurons
> (P < 0.001, chi-squared test).

Criterion: labelled PT and IT cells imaged 450 to 700 um deep; selectivity and preference as
imaging-selectivity-counts, with its readings. Reproduced when, under one reading, the PT binomial
test (against 0.5, two-sided; one-sided reported) gives P < 0.001 with a contralateral majority,
the IT test P > 0.05, the chi-squared test of PT against IT P < 0.001, and the counts are within
tolerance; partially reproduced when the three tests come out so and a value falls outside. The
files give no plane depth, so every included cell labelled PT or IT is used.

In [ ]:
types = np.concatenate([pl["roi_type"] for pl in O])
inc = np.concatenate([pl["included"] for pl in O])
rows = []
for key, res in IMG.items():
    S = np.concatenate([r["sel"] for r in res]); C = np.concatenate([r["contra"] for r in res])
    pt, it = (types == "PT") & inc, (types == "IT") & inc
    kp, np_ = int((pt & S & C).sum()), int((pt & S).sum())
    ki, ni = int((it & S & C).sum()), int((it & S).sum())
    tab = [[kp, np_ - kp], [ki, ni - ki]]
    rows.append(dict(response=key[0], span=key[1], rank=key[2], trials=key[3], PT=int(pt.sum()), PT_sel=np_,
                     PT_contra=round(kp / np_, 3), IT=int(it.sum()), IT_sel=ni, IT_contra=round(ki / ni, 3),
                     P_PT=stats.binomtest(kp, np_, 0.5).pvalue, P_PT_1sided=stats.binomtest(kp, np_, 0.5, alternative="greater").pvalue,
                     P_IT=stats.binomtest(ki, ni, 0.5).pvalue, P_chi2=stats.chi2_contingency(tab).pvalue,
                     P_chi2_nocorr=stats.chi2_contingency(tab, correction=False).pvalue))
H = pd.DataFrame(rows)
H["tests_ok"] = (H.P_PT < 0.001) & (H.PT_contra > 0.5) & (H.P_IT > 0.05) & (H.P_chi2 < 0.001)
for col, (v, t) in dict(PT=(534, 53.4), IT=(542, 54.2), PT_sel=(182, 18.2), IT_sel=(174, 17.4), PT_contra=(0.67, 0.067), IT_contra=(0.48, 0.048)).items():
    H[col + "_ok"] = (H[col] - v).abs() <= t + 1e-9
print(H.round(4).to_string(index=False))
print("Readings where the three tests come out as in the paper:", int(H.tests_ok.sum()), "of", len(H))
print("Readings with PT binomial P < 0.001:", int((H.P_PT < 0.001).sum()), "; with chi-squared P < 0.001:", int((H.P_chi2 < 0.001).sum()),
      "; smallest chi-squared P:", round(H.P_chi2.min(), 4))
print("Labelled cells by mouse (included):")
print(pd.crosstab(np.concatenate([[pl["subject"]] * len(pl["roi_type"]) for pl in O])[inc], types[inc]).to_string())

**Result: not reproduced.** Without the layer 5 selection, 530 included cells are labelled PT
(paper 534) and 843 IT (paper 542). Selective PT cells are contra-preferring in a majority under
every reading (0.577 to 0.649; binomial P < 0.001 in 5 of 16 readings) and IT cells are not (0.476 to
0.532; P > 0.05 in all). The chi-squared test of PT against IT never reaches P < 0.001 (smallest
0.0016), so in no reading do the three tests come out as in the paper. The direction of the
finding holds; the IT counts suggest that IT cells outside layer 5 are included.

## Summary

| Claim | Outcome |
|---|---|
| behaviour-performance | reproduced |
| fig1d-delay-inhibition-ipsi-bias | blocked: data missing |
| fig1d-sample-inhibition-smaller | blocked: data missing |
| units-classification | partially reproduced |
| fig2-selective-fraction | reproduced |
| fig2d-preparatory-perimovement | not reproduced |
| fig2c-selectivity-time-course | reproduced |
| fig2c-population-dynamics | blocked: methods underspecified |
| ed2c-error-trials-predict-choice | reproduced |
| fig2e-no-contra-bias-spikes | reproduced |
| ed1-fast-spiking-properties | partially reproduced |
| tagging-counts | partially reproduced |
| antidromic-latencies | blocked: methods underspecified |
| fig3e-pt-it-both-activity-types | reproduced |
| fig3f-pt-contra-preferring | blocked: methods underspecified |
| fig3f-pt-contra-selectivity | not reproduced |
| imaging-selectivity-counts | partially reproduced |
| fig4f-spatially-intermingled | not reproduced |
| fig4h-pt-contra-imaging | not reproduced |
| reaction-time | blocked: methods underspecified |
| fig5b-pt-delay-activation-contra-bias | blocked: data missing |
| fig5b-pt-sample-activation-contra-bias | blocked: data missing |
| ed6b-pt-dose-response | blocked: data missing |
| ed6d-it-activation-inconsistent | blocked: data missing |
| ed3e-high-power-contra-licking | blocked: data missing |
| ed7-pt-activation-persistent | blocked: data missing |
| ed3-anatomy-it-pt-separate | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The behaviour uses the 23 mice of the dandiset, not
the paper's 42, and only trials with events; early licks are flagged only in the extracellular
files, and trials with tagging light are left out. The electrophysiology claims are run with and
without the trials with light, which the claims file does not mention. PT/IT duplicates are
counted once. Several method details the claims file leaves open are
run under every reading it or the questions name; where readings differ in outcome the notes say
which ones match. The imaging analysis cannot select layer 5 cells and uses all labelled cells; the
inclusion flag of the files stands in for the Methods' brightness rule. Spatial distances are in
pixels.

What was found about the dataset. The dandiset holds the paper's 99 extracellular sessions
(19 mice) and 59 imaging planes (4 mice, 4,706 included cells), with 1,245 putative pyramidal and
124 fast-spiking units and the 45 PT and 27 IT labels. It holds none of the photoinhibition and
photoactivation data (Figures 1 and 5, Extended Data Figures 3, 6 and 7) and no histology. It holds
the tagging light of 75 sessions as laser power traces, delivered during behavioural trials that
the trial tables do not flag; the traces are labelled watts but hold milliwatts, and the two Tlx x
Ai32 tagging mice have none. Every PT and IT unit is deposited twice (as PT or IT and as Pyr). The 39 intermediate-width units are absent.
The extracellular trial tables hold 1,540 trials with a lick instruction that carry no events;
1,447 are marked "right", "hit" and "no early" and 1,345 of these hold no spikes. Taken at face
value they inflate performance. The imaging files flag no early licks, give no plane depth, and label the imaging mice
"Rosa-lsl-tdtomato" where the paper names C57Bl/6Crl. The lick trace is present but undocumented
as the reaction time signal.